In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# ==================================================
# Config
# ==================================================
FILE_PATH = "session_dataset.csv"
TARGET = "any_addon_added"

# keep targeting to a realistic small fraction of traffic
MIN_TARGET_FRACTION = 0.15
MAX_TARGET_FRACTION = 0.20

# ==================================================
# 1. Load data
# ==================================================
df = pd.read_csv(FILE_PATH)

print("Original shape:", df.shape)

# ==================================================
# 2. Drop leakage + IDs + raw lookup-like columns
# ==================================================
leakage_cols = [
    "actual_added_addon_names",
    "actual_added_addon_categories",
    "actual_added_addon_count",
    "actual_added_addon_value",
    "final_order_value",
    "cart_completion_score",
]

id_like_cols = [
    "session_id",
    "session_timestamp",
    "user_id",
    "restaurant_id",
    "restaurant_name",
]

raw_name_cols = [
    "base_cart_item_names",
    "recommended_addon_names",
]

drop_cols = [c for c in (leakage_cols + id_like_cols + raw_name_cols) if c in df.columns]

print("\nDropping columns:")
for c in drop_cols:
    print("-", c)

df = df.drop(columns=drop_cols)

print("\nShape after dropping columns:", df.shape)

# ==================================================
# 3. Split X / y
# ==================================================
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

X = df.drop(columns=[TARGET]).copy()
y = pd.to_numeric(df[TARGET], errors="coerce").fillna(0).astype(int)

print("\nTarget distribution:")
print(y.value_counts(dropna=False))
print("Overall conversion rate:", round(y.mean(), 4))

# ==================================================
# 4. Identify column types
# ==================================================
numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("\nNumeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

# ==================================================
# 5. Train / test split
# ==================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ==================================================
# 6. Preprocessing
# ==================================================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# ==================================================
# 7. Model
# ==================================================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# ==================================================
# 8. Train
# ==================================================
pipeline.fit(X_train, y_train)

# ==================================================
# 9. Score
# ==================================================
probs = pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, probs)
pr_auc = average_precision_score(y_test, probs)

print("\n=== Model quality ===")
print("ROC AUC:", round(roc_auc, 4))
print("PR AUC :", round(pr_auc, 4))

# ==================================================
# 10. Threshold search
# Goal:
#   choose a realistic threshold that yields
#   a small but non-trivial fraction of traffic
#   while maximizing conversion rate
# ==================================================
candidate_thresholds = np.unique(np.round(probs, 6))
candidate_thresholds = np.sort(candidate_thresholds)

results = []

overall_conversion = float(y_test.mean())
n_test = len(y_test)

for thr in candidate_thresholds:
    mask = probs >= thr
    frac_selected = float(mask.mean())
    n_selected = int(mask.sum())

    if n_selected == 0:
        continue

    # enforce realistic traffic band
    if frac_selected < MIN_TARGET_FRACTION:
        continue
    if frac_selected > MAX_TARGET_FRACTION:
        continue

    targeted_conversion = float(y_test[mask].mean())
    abs_lift = float(targeted_conversion - overall_conversion)
    rel_lift = float(targeted_conversion / overall_conversion) if overall_conversion > 0 else np.nan

    results.append({
        "threshold": float(thr),
        "sessions_selected": n_selected,
        "fraction_selected": frac_selected,
        "targeted_conversion": targeted_conversion,
        "overall_conversion": overall_conversion,
        "absolute_lift": abs_lift,
        "relative_lift": rel_lift,
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError(
        f"No valid threshold found between {MIN_TARGET_FRACTION:.2f} and {MAX_TARGET_FRACTION:.2f} of traffic."
    )

# choose threshold with highest targeted conversion
# break ties by selecting larger traffic share, then higher threshold
best_row = results_df.sort_values(
    ["targeted_conversion", "fraction_selected", "threshold"],
    ascending=[False, False, False]
).iloc[0]

best_threshold = float(best_row["threshold"])
best_fraction = float(best_row["fraction_selected"])
best_n = int(best_row["sessions_selected"])
best_targeted_conversion = float(best_row["targeted_conversion"])
best_abs_lift = float(best_row["absolute_lift"])
best_rel_lift = float(best_row["relative_lift"])

print("\n=== Best targeting strategy under traffic band ===")
print("Threshold:", round(best_threshold, 4))
print("Proportion selected:", round(best_fraction, 4))
print("Sessions selected:", best_n)
print("Overall conversion:", round(overall_conversion, 4))
print("Targeted conversion:", round(best_targeted_conversion, 4))
print("Absolute lift:", round(best_abs_lift, 4))
print("Relative lift:", round(best_rel_lift, 4))

# ==================================================
# 11. Exact top-20% reference strategy
# ==================================================
top20_threshold = float(np.percentile(probs, 80))
top20_mask = probs >= top20_threshold
top20_fraction = float(top20_mask.mean())
top20_n = int(top20_mask.sum())
top20_conversion = float(y_test[top20_mask].mean())
top20_abs_lift = float(top20_conversion - overall_conversion)
top20_rel_lift = float(top20_conversion / overall_conversion) if overall_conversion > 0 else np.nan

print("\n=== Reference top-20% strategy ===")
print("Threshold:", round(top20_threshold, 4))
print("Proportion selected:", round(top20_fraction, 4))
print("Sessions selected:", top20_n)
print("Overall conversion:", round(overall_conversion, 4))
print("Targeted conversion:", round(top20_conversion, 4))
print("Absolute lift:", round(top20_abs_lift, 4))
print("Relative lift:", round(top20_rel_lift, 4))

# ==================================================
# 12. Save scored sessions
# ==================================================
scored_test = X_test.copy()
scored_test[TARGET] = y_test.values
scored_test["predicted_probability"] = probs
scored_test["target_aggressive_reco_best"] = (probs >= best_threshold).astype(int)
scored_test["target_aggressive_reco_top20"] = (probs >= top20_threshold).astype(int)
scored_test = scored_test.sort_values("predicted_probability", ascending=False)

scored_test.to_csv("q4_scored_test_sessions.csv", index=False)
results_df.to_csv("q4_threshold_search_results.csv", index=False)

# ==================================================
# 13. Graphs for report
# ==================================================

# targeted conversion vs fraction selected
plot_df = results_df.sort_values("fraction_selected")

plt.figure(figsize=(10, 6))
plt.plot(plot_df["fraction_selected"], plot_df["targeted_conversion"], marker="o")
plt.axhline(overall_conversion, linestyle="--", label="Overall conversion")
plt.xlabel("Fraction of Sessions Targeted")
plt.ylabel("Conversion Rate in Targeted Sessions")
plt.title("Targeted Conversion vs Fraction of Traffic Selected")
plt.legend()
plt.tight_layout()
plt.savefig("q4_targeted_conversion_vs_fraction.png", dpi=200)
plt.close()

# lift vs fraction selected
plt.figure(figsize=(10, 6))
plt.plot(plot_df["fraction_selected"], plot_df["absolute_lift"], marker="o")
plt.axhline(0, linestyle="--")
plt.xlabel("Fraction of Sessions Targeted")
plt.ylabel("Absolute Lift Over Baseline")
plt.title("Absolute Lift vs Fraction of Traffic Selected")
plt.tight_layout()
plt.savefig("q4_lift_vs_fraction.png", dpi=200)
plt.close()

# predicted probability distribution
plt.figure(figsize=(10, 6))
plt.hist(probs, bins=40)
plt.axvline(best_threshold, linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.axvline(top20_threshold, linestyle=":", label=f"Top-20% threshold = {top20_threshold:.3f}")
plt.xlabel("Predicted Probability")
plt.ylabel("Number of Sessions")
plt.title("Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.savefig("q4_probability_distribution.png", dpi=200)
plt.close()

# ==================================================
# 14. Reference summary
# ==================================================
summary = {
    "overall_conversion": overall_conversion,
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "best_threshold": best_threshold,
    "best_fraction_selected": best_fraction,
    "best_sessions_selected": best_n,
    "best_targeted_conversion": best_targeted_conversion,
    "best_absolute_lift": best_abs_lift,
    "best_relative_lift": best_rel_lift,
    "top20_threshold_reference": top20_threshold,
    "top20_fraction_reference": top20_fraction,
    "top20_sessions_reference": top20_n,
    "top20_conversion_reference": top20_conversion,
    "top20_absolute_lift_reference": top20_abs_lift,
    "top20_relative_lift_reference": top20_rel_lift,
}

print("\n=== Reference summary ===")
print(summary)

print("\nSaved files:")
print("- q4_scored_test_sessions.csv")
print("- q4_threshold_search_results.csv")
print("- q4_targeted_conversion_vs_fraction.png")
print("- q4_lift_vs_fraction.png")
print("- q4_probability_distribution.png")

Original shape: (100000, 57)

Dropping columns:
- actual_added_addon_names
- actual_added_addon_categories
- actual_added_addon_count
- actual_added_addon_value
- final_order_value
- cart_completion_score
- session_id
- session_timestamp
- user_id
- restaurant_id
- restaurant_name
- base_cart_item_names
- recommended_addon_names

Shape after dropping columns: (100000, 44)

Target distribution:
any_addon_added
0    76378
1    23622
Name: count, dtype: int64
Overall conversion rate: 0.2362

Numeric columns: 28
Categorical columns: 15

Train shape: (80000, 43)
Test shape: (20000, 43)

=== Model quality ===
ROC AUC: 0.585
PR AUC : 0.2952

=== Best targeting strategy under traffic band ===
Threshold: 0.5007
Proportion selected: 0.1504
Sessions selected: 3008
Overall conversion: 0.2362
Targeted conversion: 0.3295
Absolute lift: 0.0933
Relative lift: 1.3948

=== Reference top-20% strategy ===
Threshold: 0.5005
Proportion selected: 0.2
Sessions selected: 4000
Overall conversion: 0.2362
Targete